In [1]:
import os
import urllib.request
import torch

In [2]:
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "gpt_data/shakespeare.txt"

In [3]:
def download_shakespeare():
    """Download the Shakespeare dataset if not already present."""
    if os.path.exists(DATA_PATH):
        print(f"Dataset already exists at {DATA_PATH}")
        return

    print("Downloading Shakespeare dataset...")
    os.makedirs("gpt_data", exist_ok=True)
    urllib.request.urlretrieve(SHAKESPEARE_URL, DATA_PATH)
    print(f"Downloaded to {DATA_PATH}")

In [4]:
download_shakespeare()

Downloaded to gpt_data/shakespeare.txt


In [5]:
class CharacterTokenizer():
  def __init__(self, text: str):
    self.characters = sorted(list(set(text)))

    self.vocab_size = len(self.characters)

    self.char_to_id = {}
    for index, char in enumerate(self.characters):
      self.char_to_id[char] = index

    self.id_to_char = {}
    for index, char in enumerate(self.characters):
      self.id_to_char[index] = char

    print(f"vocabulary size: {self.vocab_size} characters")
    print(f"Characters: {repr(''.join(self.characters[:50]))}...")

  def encode(self, text: str) -> list:
    ids = []
    for char in text:
      ids.append(self.char_to_id[char])
    return ids

  def decode(self, ids: list) -> str:
    if isinstance(ids, torch.Tensor):
      ids = ids.tolist()

    characters = []
    for id in ids:
      characters.append(self.id_to_char[id])
    return ''.join(characters)

In [6]:
def load_data(train_split: float = 0.9):
    download_shakespeare()

    with open(DATA_PATH, 'r', encoding='utf-8') as file:
        text = file.read()

    tokenizer = CharacterTokenizer(text)

    all_ids = tokenizer.encode(text)
    data = torch.tensor(all_ids, dtype=torch.long)

    split_index = int(train_split * len(data))
    train_data = data[:split_index]
    val_data = data[split_index:]

    print(f"\ntrain size: {len(train_data):,} tokens")
    print(f"test size: {len(val_data):,} tokens")

    return train_data, val_data, tokenizer

In [7]:
def get_batch(data: torch.Tensor, block_size: int, batch_size: int):
  max_start = len(data) - block_size -1
  positions = torch.randint(low=0, high=max_start, size=(batch_size,))

  x_list = []
  y_list = []

  for pos in positions:
    x_list.append(data[pos : pos+block_size])
    y_list.append(data[pos+1 : pos+block_size+1])

  x = torch.stack(x_list)
  y = torch.stack(y_list)

  return x, y

In [8]:
import torch.nn as nn
import torch.nn.functional as F

In [9]:
class TransformerBlock(nn.Module):
  def __init__(self,
               embedding_dim: int=384,
               num_heads: int=6,
               dropout: float=0.1):
    super().__init__()

    self.ln1 = nn.LayerNorm(embedding_dim)

    self.attention = nn.MultiheadAttention(
      embed_dim=embedding_dim,
      num_heads=num_heads,
      dropout=dropout,
      batch_first=True
    )

    self.ln2 = nn.LayerNorm(embedding_dim)

    self.mlp = nn.Sequential(
      nn.Linear(embedding_dim, 4 * embedding_dim),
      nn.GELU(),
      nn.Linear(4 * embedding_dim, embedding_dim),
      nn.Dropout(dropout)
    )

  def forward(self, x: torch.Tensor, causal_mask: torch.Tensor) -> torch.Tensor:
    x_norm = self.ln1(x)

    attn_output, _ = self.attention(query=x_norm,
                                    key=x_norm,
                                    value=x_norm,
                                    attn_mask=causal_mask,
                                    is_causal=False
    )

    x = x + attn_output
    x = x + self.mlp(self.ln2(x))

    return x

In [10]:
class GPT(nn.Module):
  def __init__(self,
               vocab_size: int,
               embedding_dim: int = 384,
               num_heads: int = 6,
               num_layers: int = 6,
               block_size: int = 256,
               dropout: float= 0.1):
    super().__init__()

    self.block_size = block_size

    self.token_embedding = nn.Embedding(
      num_embeddings=vocab_size,
      embedding_dim=embedding_dim
    )

    self.position_embedding = nn.Embedding(
      num_embeddings=block_size,
      embedding_dim=embedding_dim
    )

    self.dropout = nn.Dropout(dropout)

    self.blocks = nn.ModuleList([
        TransformerBlock(embedding_dim=embedding_dim,
                         num_heads=num_heads,
                         dropout=dropout
                         ) for _ in range(num_layers)
    ])

    self.ln_final = nn.LayerNorm(embedding_dim)

    self.output_proj = nn.Linear(embedding_dim, vocab_size)

    self.loss_fn = nn.CrossEntropyLoss()

    casual_mask = torch.triu(
      torch.ones(block_size, block_size, dtype=torch.bool),
      diagonal=1
    )

    self.register_buffer("casual_mask", casual_mask)

    self.apply(self._init_weights)

    total_params = sum(p.numel() for p in self.parameters())
    print(f"GPT model created with {total_params:,} parameters")

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  def forward(self,
              input_ids: torch.Tensor,
              targets: torch.Tensor = None):
    batch_size, seq_len = input_ids.shape
    device = input_ids.device

    token_emb = self.token_embedding(input_ids)

    positions = torch.arange(seq_len, device=device)

    pos_emb = self.position_embedding(positions)

    x = self.dropout(token_emb + pos_emb)

    mask = self.casual_mask[:seq_len, :seq_len]

    for block in self.blocks:
      x = block(x, mask)

    x = self.ln_final(x)

    logits = self.output_proj(x)

    loss = None
    if targets is not None:

      batch_size, seq_len, vocab_size = logits.shape

      logits_flat = logits.reshape(batch_size * seq_len, vocab_size)
      targets_flat = targets.reshape(batch_size * seq_len)

      loss = self.loss_fn(logits_flat, targets_flat)

    return logits, loss
  @torch.no_grad()
  def generate(self, input_ids: torch.Tensor, max_new_tokens: int, temperature: float = 1.0):
    for _ in range(max_new_tokens):
      if input_ids.size(1) <= self.block_size:
        current_input = input_ids
      else:
        current_input = input_ids[:, -self.block_size:]

      logits, _ = self.forward(current_input)
      last_logits = logits[:, -1, :]

      last_logits = last_logits / temperature

      probs = F.softmax(last_logits, dim=-1)

      next_token = torch.multinomial(probs, num_samples=1)

      input_ids = torch.cat([input_ids, next_token], dim=1)
    return input_ids


In [11]:
import time

EMBEDDING_DIM = 384
NUM_HEADS = 6
NUM_LAYERS = 6
BLOCK_SIZE = 256
DROPOUT = 0.1

BATCH_SIZE = 64
MAX_ITERS = 5000
EVAL_INTERVAL = 500
LEARNING_RATE = 3e-4
WARMUP_ITERS = 100

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_PATH = "checkpoints/model.pt"

In [12]:
def get_learning_rate(iteration):
    if iteration < WARMUP_ITERS:
        return LEARNING_RATE * (iteration / WARMUP_ITERS)
    else:
        return LEARNING_RATE

@torch.no_grad()
def evaluate(model, train_data, val_data):
  model.eval()

  results = {}
  for name, data in [("train", train_data), ("val", val_data)]:
    total_loss = 0.0

    for _ in range(100):
      x, y = get_batch(data, BLOCK_SIZE, BATCH_SIZE)
      x, y = x.to(DEVICE), y.to(DEVICE)
      _, loss = model(x, y)
      total_loss += loss.item()

    results[name] = total_loss / 100

  model.train()
  return results

@torch.no_grad()
def generate_sample(model, tokenizer):
  model.eval()

  prompt = "ROMEO:"
  prompt_ids = tokenizer.encode(prompt)
  input_ids = torch.tensor(prompt_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

  output_ids = model.generate(input_ids, max_new_tokens=200, temperature=0.8)

  model.train()
  return tokenizer.decode(output_ids[0])

def train():
    train_data, val_data, tokenizer = load_data()
    vocab_size = tokenizer.vocab_size

    model = GPT(
        vocab_size=vocab_size,
        embedding_dim=EMBEDDING_DIM,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        block_size=BLOCK_SIZE,
        dropout=DROPOUT
    )
    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    os.makedirs("checkpoints", exist_ok=True)
    start_time = time.time()

    for iteration in range(MAX_ITERS):
        lr = get_learning_rate(iteration)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        x, y = get_batch(train_data, BLOCK_SIZE, BATCH_SIZE)
        x, y = x.to(DEVICE), y.to(DEVICE)

        _, loss = model(x, y)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        if iteration % EVAL_INTERVAL == 0 or iteration == MAX_ITERS - 1:
          losses = evaluate(model, train_data, val_data)
          elapsed = time.time() - start_time

          print(f"Iter {iteration:5d} | "
                f"Train Loss: {losses['train']:.4f} | "
                f"Val Loss: {losses['val']:.4f} | "
                f"LR: {lr:.2e} | "
                f"Time: {elapsed:.0f}s")

          if iteration > 0:
            print("\n--- Sample ---")
            print(generate_sample(model, tokenizer)[:400])
            print("--------------\n")

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'iteration': MAX_ITERS,
        'val_loss': losses['val'],
        'config': {
            'vocab_size': vocab_size,
            'embedding_dim': EMBEDDING_DIM,
            'num_heads': NUM_HEADS,
            'num_layers': NUM_LAYERS,
            'block_size': BLOCK_SIZE,
        }
    }

    torch.save(checkpoint, CHECKPOINT_PATH)
    total_time = time.time() - start_time
    print("\n" + "=" * 60)
    print("Training Complete!")
    print("=" * 60)
    print(f"Total time: {total_time / 60:.1f} minutes")
    print(f"Final val loss: {losses['val']:.4f}")
    print(f"Model saved to: {CHECKPOINT_PATH}")


In [ ]:
if os.path.exists(CHECKPOINT_PATH):
    print("Pretrained model found!")
    print("Skipping training.")
else:
    print("No pretrained model found.")
    print("Starting training...")
    train()

Dataset already exists at gpt_data/shakespeare.txt
vocabulary size: 65 characters
Characters: "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijk"...

train size: 1,003,854 tokens
test size: 111,540 tokens
GPT model created with 10,795,841 parameters
Iter     0 | Train Loss: 4.3025 | Val Loss: 4.3054 | LR: 0.00e+00 | Time: 32s
Iter   500 | Train Loss: 1.6988 | Val Loss: 1.8502 | LR: 3.00e-04 | Time: 334s

--- Sample ---
ROMEO:
I he my good look not prince;
And shall the come and this so made day a their grie.

QUEEN ELIZABETH:
Nay, what I but are with my ove.

BENVOLIO:
The is the must should the see.

POLINUS:
Parius it f
--------------

Iter  1000 | Train Loss: 1.3623 | Val Loss: 1.5856 | LR: 3.00e-04 | Time: 640s

--- Sample ---
ROMEO:
What is so colam? that you not shall you do.

AUTOLYCUS:
Aland the would the lies.

PAULINA:
Conscess their was them sins with him?

CAMILLO:
I will not knock,
What shall petter to humbly mouth?

COR
--------------

Iter  1500 | Train Loss: 1.2219 | 

In [15]:
def load_model(checkpoint_path: str = "checkpoints/model.pt"):
  device = "mps" if torch.backends.mps.is_available() else "cpu"

  checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

  config = checkpoint['config']

  download_shakespeare()
  with open(DATA_PATH, 'r', encoding='utf-8') as file:
      text = file.read()
  tokenizer = CharacterTokenizer(text)

  model = GPT(
      vocab_size=config['vocab_size'],
      embedding_dim=config['embedding_dim'],
      num_heads=config['num_heads'],
      num_layers=config['num_layers'],
      block_size=config['block_size'],
      dropout=0.0
  )

  model.load_state_dict(checkpoint['model_state_dict'])

  model = model.to(device)
  model.eval()

  return model, tokenizer, device

@torch.no_grad()
def generate(model, tokenizer, device, prompt: str, max_tokens: int = 500, temperature: float = 0.8):

  prompt_ids = tokenizer.encode(prompt)
  input_ids = torch.tensor(prompt_ids, dtype=torch.long, device=device)
  input_ids = input_ids.unsqueeze(0)

  output_ids = model.generate(
      input_ids=input_ids,
      max_new_tokens=max_tokens,
      temperature=temperature
  )

  generated_text = tokenizer.decode(output_ids[0])

  return generated_text

In [17]:
model, tokenizer, device = load_model("checkpoints/model.pt")

# define some example prompts to try
prompts = [
    "ROMEO:",
    "To be, or not to be",
    "JULIET:\nO Romeo, ",
]

# generate text for each prompt
print("\n" + "=" * 60)
print("Generating Text...")
print("=" * 60)

for prompt in prompts:
  print(f"\n--- Prompt: {repr(prompt)} ---\n")

  generated_text = generate(
      model=model,
      tokenizer=tokenizer,
      device=device,
      prompt=prompt,
      max_tokens=300,
      temperature=0.8
  )

  print(generated_text)
  print("\n" + "-" * 60)

# interactive: let user try their own prompts
print("\n" + "=" * 60)
print("Try Your Own Prompts!")
print("=" * 60)
print("Enter a prompt to generate text, or 'quit' to exit.")
print("Tip: Try character names like 'HAMLET:', 'KING:', 'First Citizen:'")

while True:
  try:
    prompt = input("\nYour prompt: ")

    if prompt.lower() in ['quit', 'exit', 'q']:
      print("Farewell!")
      break

    if not prompt:
      continue

    generated_text = generate(
        model=model,
        tokenizer=tokenizer,
        device=device,
        prompt=prompt,
        max_tokens=500,
        temperature=0.8
    )

    print("\n" + generated_text)

  except KeyboardInterrupt:
    print("\n\nFarewell!")
    break

Dataset already exists at gpt_data/shakespeare.txt
vocabulary size: 65 characters
Characters: "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijk"...
GPT model created with 10,795,841 parameters

Generating Text...

--- Prompt: 'ROMEO:' ---

ROMEO:
Why, how now, change prince! Few what do you all!
The children of an unnally strength,
Come but with a tear-weather-weapoor shine.
Love gaves your shame: the sweet joy
And strength unto the lord pronounce and sell the
duke's wife to distress him; he is come too late;
much more than a vice betrous n

------------------------------------------------------------

--- Prompt: 'To be, or not to be' ---

To be, or not to be a poor petitioner.

SICINIUS:
This is a prophecy goodly to to the Tower.
Hast any bond to save his life of his wife, his
father eyes were they gave to take his leave.

SICINIUS:
Nature need; neither made you are well consent
To be your lady's blood. When you have been in this place
Whom envious cou

---------------------------